# Mnemonics LongMemEval — 50q + 500q (Colab)

**Amaç:** assistant_facts patch'inin gerçek lift'ini ölç.

Drive'a şu 3 dosya yüklenmiş olmalı `My Drive/lme/` altında:
- `longmemeval_s_cleaned.json` (264 MB, dataset)
- `entity_graph_result.json` (148 KB, MemPalace baseline)
- `adaptmem-model/` klasörü (87 MB, FT encoder — opsiyonel)

Çalıştırma sırası: hücreleri yukarıdan aşağı sırayla. T4 GPU önerilir (Runtime → Change runtime type → T4).

## 1) Repo + bağımlılıklar

In [ ]:
!git clone https://github.com/nakata-app/mnemonics.git /content/mnemonics
%cd /content/mnemonics
!git log --oneline -3

In [ ]:
!pip install -q -e . sentence-transformers numpy 2>&1 | tail -5

## 2) Drive mount + path patch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DATA = '/content/drive/MyDrive/lme/longmemeval_s_cleaned.json'
MEMPALACE = '/content/drive/MyDrive/lme/entity_graph_result.json'
ADAPTMEM = '/content/drive/MyDrive/lme/adaptmem-model'

print('DATA exists:', os.path.exists(DATA))
print('MEMPALACE exists:', os.path.exists(MEMPALACE))
print('ADAPTMEM exists:', os.path.isdir(ADAPTMEM))

In [ ]:
# Path patch: harness'taki sabit Mac path'lerini Drive path'lerine çevir
import re
p = pathlib.Path('/content/mnemonics/benchmarks/longmemeval_eval.py')
src = p.read_text()
src = re.sub(r'DATA = Path\([^)]+\)', f'DATA = Path("{DATA}")', src)
src = re.sub(r'MEMPALACE_BASELINE = Path\([\s\S]*?\)', f'MEMPALACE_BASELINE = Path("{MEMPALACE}")', src)
p.write_text(src)
!grep -n 'DATA = Path\|MEMPALACE_BASELINE = Path' {p}

In [ ]:
# AdaptMem opsiyonel: varsa kullan, yoksa default MiniLM
if os.path.isdir(ADAPTMEM) and os.path.exists(os.path.join(ADAPTMEM, 'model.safetensors')):
    os.environ['MNEMONICS_ADAPTMEM_PATH'] = ADAPTMEM
    print('Using AdaptMem FT:', ADAPTMEM)
else:
    print('AdaptMem yok — default MiniLM ile koşacak')

# Sonuçlar Drive'a yazılacak
RESULTS = '/content/drive/MyDrive/lme/results'
os.makedirs(RESULTS, exist_ok=True)
print('Results dir:', RESULTS)

## 3) Smoke test (n=5, hızlı bakış)

In [ ]:
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 5 --mode no_rerank \
    --augment-preferences --augment-assistant-facts --candidate-k 50 \
    --out /tmp/smoke.json

## 4) 50q eval — baseline (facts OFF) vs facts (ON)

Aynı seed=42 ile iki koşum. Delta = facts'in net katkısı (50q apples-to-apples).

In [ ]:
# 4A) baseline: facts OFF (mevcut yayında olan ayar — augment_prefs + cand_k=50)
import datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 50 --mode rerank \
    --augment-preferences --candidate-k 50 \
    --out {RESULTS}/lme50_baseline_{ts}.json \
    --per-q-out {RESULTS}/lme50_baseline_perq_{ts}.json 2>&1 | tee {RESULTS}/lme50_baseline_{ts}.log

In [ ]:
# 4B) facts ON
ts2 = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 50 --mode rerank \
    --augment-preferences --augment-assistant-facts --candidate-k 50 \
    --out {RESULTS}/lme50_facts_{ts2}.json \
    --per-q-out {RESULTS}/lme50_facts_perq_{ts2}.json 2>&1 | tee {RESULTS}/lme50_facts_{ts2}.log

In [ ]:
# 4C) delta özet
import json, glob
def latest(prefix):
    files = sorted(glob.glob(f'{RESULTS}/{prefix}_2*.json'))
    return json.load(open(files[-1]))

b = latest('lme50_baseline')['mnemonics_rerank']
f = latest('lme50_facts')['mnemonics_rerank']
for k in ('R@1', 'R@5', 'R@10'):
    print(f'{k:6}  baseline={b[k]:.3f}  facts={f[k]:.3f}  Δ={f[k]-b[k]:+.3f}')

In [ ]:
# 4D) hangi sorular kurtarıldı? per-q diff
def latest_perq(prefix):
    files = sorted(glob.glob(f'{RESULTS}/{prefix}_perq_*.json'))
    return {r['qid']: r for r in json.load(open(files[-1]))}

pb = latest_perq('lme50_baseline')
pf = latest_perq('lme50_facts')
rescued, broken = [], []
for qid in pb:
    if qid not in pf: continue
    if pf[qid]['hit@10'] and not pb[qid]['hit@10']:
        rescued.append((qid, pb[qid]['qtype'], pb[qid]['question'][:100]))
    elif pb[qid]['hit@10'] and not pf[qid]['hit@10']:
        broken.append((qid, pb[qid]['qtype'], pb[qid]['question'][:100]))

print(f'RESCUED ({len(rescued)}): facts ile yakalandı, baseline kaçırmıştı')
for qid, t, q in rescued:
    print(f'  [{t}] {qid}: {q}')
print(f'\nBROKEN ({len(broken)}): facts ile kayboldu, baseline yakalamıştı')
for qid, t, q in broken:
    print(f'  [{t}] {qid}: {q}')

## 5) 500q full eval — baseline + facts paralel değil, sıralı

T4 GPU'da rerank 500q ≈ 90-120 dk her biri. Toplam ~3-4 saat.  
Colab session timeout'una dikkat (interactive 12 saat, idle 90 dk).

In [ ]:
# 5A) 500q baseline (facts OFF)
ts3 = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --candidate-k 50 \
    --out {RESULTS}/lme500_baseline_{ts3}.json \
    --per-q-out {RESULTS}/lme500_baseline_perq_{ts3}.json 2>&1 | tee {RESULTS}/lme500_baseline_{ts3}.log

In [ ]:
# 5B) 500q facts ON
ts4 = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --augment-assistant-facts --candidate-k 50 \
    --out {RESULTS}/lme500_facts_{ts4}.json \
    --per-q-out {RESULTS}/lme500_facts_perq_{ts4}.json 2>&1 | tee {RESULTS}/lme500_facts_{ts4}.log

In [ ]:
# 5C) 500q delta + type breakdown
b500 = latest('lme500_baseline')['mnemonics_rerank']
f500 = latest('lme500_facts')['mnemonics_rerank']
print('=== OVERALL ===')
for k in ('R@1','R@5','R@10'):
    print(f'{k:6}  baseline={b500[k]:.3f}  facts={f500[k]:.3f}  Δ={f500[k]-b500[k]:+.3f}')
print('\n=== BY TYPE ===')
for qt in b500['by_type']:
    bb = b500['by_type'][qt]
    ff = f500['by_type'].get(qt, {})
    print(f'{qt:20} n={bb["n"]:3}  R@10 base={bb["R@10"]:.3f}  facts={ff.get("R@10",0):.3f}  Δ={ff.get("R@10",0)-bb["R@10"]:+.3f}')